# V2 classified-paragraph extraction: smoke and full processing

This Colab notebook consumes documents that are already split and classified by paragraph in `classification.parquet`. It does not download, decode, or parse raw RTF documents. First validate the production prompts with isolated local smoke artifacts, then optionally promote the same prompts to the resumable full-dataset cloud pipeline.

## Dependencies

Use the project environment locally. In a clean Colab GPU runtime, uncomment and run the installation command.

In [ ]:
# Colab only:
# %pip install -q "pyarrow>=24,<25" "transformers>=5.8,<6" "accelerate>=1.13,<2" "huggingface-hub>=0.34" "json-repair>=0.50,<1" google-cloud-bigquery google-cloud-storage striprtf
# %cd legal-doc-ua

## Locate and import the project

Open the notebook from the cloned repository, or set `REPOSITORY_ROOT` to the repository mounted in Colab.

In [ ]:
from pathlib import Path
import sys

import pyarrow.parquet as pq
from IPython.display import display

REPOSITORY_ROOT = Path('/content/legal_doc_parser')
if not (REPOSITORY_ROOT / 'src' / 'document_split').is_dir():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / 'src' / 'document_split').is_dir():
            REPOSITORY_ROOT = candidate
            break

source_root = REPOSITORY_ROOT / 'src'
repository_root = REPOSITORY_ROOT
if not (source_root / 'document_split').is_dir():
    raise FileNotFoundError(
        'Repository not found. Set REPOSITORY_ROOT to the cloned or mounted project.'
    )
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))
print('Repository:', repository_root)

# Real-data extraction smoke test

The input is already paragraph-split and classified. Each document gets an isolated directory containing raw handler responses, normalized JSON, final Parquet, warnings, and a recursive schema-population report.

## Select classified real decisions

The expected layout is `<classification root>/<document id>/classification.parquet`. The default three decisions cover probation, imprisonment, civil claims, costs, physical evidence, confiscation, and security measures.

In [ ]:
from document_split.v2 import (
    DEFAULT_V2_CHAIN_SETTINGS,
    DEFAULT_V2_EXTRACTION_SETTINGS,
    DEFAULT_V2_PART_PROCESSING_PROMPTS,
    build_sample_processing_contexts,
    run_real_data_smoke,
)

# Verify the loaded Colab code handles a common model layout mistake before
# spending GPU time. Older clones corrected this only after handler validation.
from dataclasses import replace
import inspect
import pyarrow as pa
import document_split.v2.handlers as v2_handlers
from document_split import CRIMINAL_SCHEMA
from document_split.processing import normalize_known_document_layout

if not hasattr(v2_handlers, 'compose_v2_map_messages'):
    raise RuntimeError(
        'The loaded source predates grounded map/reduce. Pull or copy the '
        'updated src/document_split/v2 package, then restart the runtime.'
    )
print('Grounded map/reduce preflight: PASS')

_operative_schema = pa.schema([CRIMINAL_SCHEMA.field('operative_part')])
_layout_probe = {
    'operative_part': {
        'conviction_operative': {
            'final_sentence': 'layout-probe',
        },
    },
}
_handler_normalizer = v2_handlers.normalize_v2_payload
try:
    _handler_normalizer(_layout_probe, _operative_schema)
    _layout_fix_loaded = True
except ValueError as exc:
    _layout_fix_loaded = 'unexpected fields' not in str(exc)

if not _layout_fix_loaded:
    def _normalize_v2_payload_with_layout_fix(payload, schema):
        corrected = normalize_known_document_layout(payload, schema)
        return _handler_normalizer(corrected, schema)

    v2_handlers.normalize_v2_payload = _normalize_v2_payload_with_layout_fix
    print('Applied handler-level operative layout compatibility fix.')

_normalized_probe = v2_handlers.normalize_v2_payload(
    _layout_probe,
    _operative_schema,
)
assert (
    _normalized_probe['operative_part']['final_sentence']
    == 'layout-probe'
)
assert 'final_sentence' not in (
    _normalized_probe['operative_part']['conviction_operative']
)
print('V2 handlers loaded from:', inspect.getfile(v2_handlers))
print('Operative layout preflight: PASS')

# Older clones also exposed routing assignments for future, invisible
# paragraphs. Limit that metadata to the current targets and overlap.
import document_split.v2.sample_processing as v2_sample_processing

if not getattr(
    v2_handlers.compose_v2_handler_messages,
    '_visible_assignments_only',
    False,
):
    _compose_v2_handler_messages = v2_handlers.compose_v2_handler_messages

    def _compose_with_visible_assignments(*, state, batch, **kwargs):
        visible_ids = {
            paragraph.paragraph_id
            for paragraph in (*batch.context, *batch.targets)
        }
        original_assignments = state.part_assignments
        state.part_assignments = [
            assignment
            for assignment in original_assignments
            if assignment['paragraph_index'] in visible_ids
        ]
        try:
            return _compose_v2_handler_messages(
                state=state,
                batch=batch,
                **kwargs,
            )
        finally:
            state.part_assignments = original_assignments

    _compose_with_visible_assignments._visible_assignments_only = True
    v2_handlers.compose_v2_handler_messages = (
        _compose_with_visible_assignments
    )
    v2_sample_processing.compose_v2_handler_messages = (
        _compose_with_visible_assignments
    )
    print('Applied visible-paragraph routing compatibility fix.')
print('Batch routing metadata preflight: PASS')

# Keep valid records when an older handler clone emits or receives an
# out-of-batch paragraph index. Invalid records are removed in place before
# the original strict validator runs.
if not getattr(
    v2_handlers._validate_nested_paragraph_indexes,
    '_drops_out_of_batch_records',
    False,
):
    _strict_paragraph_index_validator = (
        v2_handlers._validate_nested_paragraph_indexes
    )

    def _drop_invalid_records_in_place(value, target_ids, path):
        if isinstance(value, dict):
            for key, child in value.items():
                _drop_invalid_records_in_place(
                    child,
                    target_ids,
                    f'{path}.{key}',
                )
        elif isinstance(value, list):
            kept = []
            for index, child in enumerate(value):
                child_path = f'{path}[{index}]'
                paragraph_index = (
                    child.get('paragraph_index')
                    if isinstance(child, dict)
                    else None
                )
                if (
                    isinstance(child, dict)
                    and 'paragraph_index' in child
                    and (
                        type(paragraph_index) is not int
                        or paragraph_index not in target_ids
                    )
                ):
                    print(
                        f'Dropped {child_path}: paragraph_index='
                        f'{paragraph_index!r} is outside '
                        f'{sorted(target_ids)}',
                        flush=True,
                    )
                    continue
                _drop_invalid_records_in_place(
                    child,
                    target_ids,
                    child_path,
                )
                kept.append(child)
            value[:] = kept

    def _validate_after_dropping(value, target_ids, *, path):
        _drop_invalid_records_in_place(value, target_ids, path)
        return _strict_paragraph_index_validator(
            value,
            target_ids,
            path=path,
        )

    _validate_after_dropping._drops_out_of_batch_records = True
    v2_handlers._validate_nested_paragraph_indexes = (
        _validate_after_dropping
    )
    print('Applied out-of-batch record filtering compatibility fix.')
print('Paragraph-index filtering preflight: PASS')

# Older clones strictly reject a model-produced list for a scalar text
# field such as legal_basis. Join only string/null arrays; arbitrary
# objects and mixed arrays remain schema errors.
import document_split.processing as document_processing

_string_probe_field = pa.field('legal_basis', pa.string())
try:
    _string_probe_result = document_processing.normalize_arrow_value(
        ['ст. 65 КК України', 'ст. 75 КК України'],
        _string_probe_field,
        'legal_basis',
    )
except TypeError:
    _string_probe_result = None

if _string_probe_result != 'ст. 65 КК України; ст. 75 КК України':
    _strict_arrow_normalizer = (
        document_processing.normalize_arrow_value
    )

    def _normalize_with_string_list(value, arrow_field, path):
        data_type = arrow_field.type
        if (
            (
                pa.types.is_string(data_type)
                or pa.types.is_large_string(data_type)
            )
            and isinstance(value, list)
            and all(
                item is None or isinstance(item, str)
                for item in value
            )
        ):
            parts = [
                item.strip()
                for item in value
                if isinstance(item, str) and item.strip()
            ]
            normalized = '; '.join(parts) if parts else None
            print(
                f'Normalized list-valued string at {path}: '
                f'{normalized!r}',
                flush=True,
            )
            return normalized
        return _strict_arrow_normalizer(value, arrow_field, path)

    document_processing.normalize_arrow_value = (
        _normalize_with_string_list
    )
    v2_handlers.normalize_arrow_value = _normalize_with_string_list
    print('Applied list-valued string compatibility fix.')

assert document_processing.normalize_arrow_value(
    ['ст. 65 КК України', 'ст. 75 КК України'],
    _string_probe_field,
    'legal_basis',
) == 'ст. 65 КК України; ст. 75 КК України'
print('String-list normalization preflight: PASS')

CLASSIFICATION_ROOT = (
    source_root / 'document_split' / 'v2' / 'document_text_parsing' / 'downloads'
)
# Start with one document (three handler calls). Add the other IDs only after
# timing and output quality are acceptable for the first document.
SMOKE_DOCUMENT_IDS = [
    '118355359',  # probation, costs, evidence
]
# Additional prepared examples:
# '118584307'  # imprisonment, civil claim
# '116798672'  # fine, confiscation, security measures

CLASSIFICATION_FILES = [
    CLASSIFICATION_ROOT / document_id / 'classification.parquet'
    for document_id in SMOKE_DOCUMENT_IDS
]
SMOKE_OUTPUT_ROOT = Path('/content/v2-real-data-smoke')
# Reasoning returns a large structured JSON object. Keep the production output
# allowance and use smaller input batches to avoid truncated responses.
SMOKE_MAX_NEW_TOKENS = 8_192
# 0.0 uses deterministic greedy decoding. A positive value enables sampling.
MODEL_TEMPERATURE = 0.0
SMOKE_TARGET_BATCH_TOKENS = 1_000
SMOKE_OVERLAP_TOKENS = 256
ACTIVE_V2_EXTRACTION_SETTINGS = replace(
    DEFAULT_V2_EXTRACTION_SETTINGS,
    max_new_tokens=SMOKE_MAX_NEW_TOKENS,
    temperature=MODEL_TEMPERATURE,
)
SMOKE_CHAIN_SETTINGS = replace(
    DEFAULT_V2_CHAIN_SETTINGS,
    target_batch_tokens=SMOKE_TARGET_BATCH_TOKENS,
    overlap_tokens=SMOKE_OVERLAP_TOKENS,
)
# Optional diagnostics for each model call.
SHOW_SMOKE_CHUNKS = False
SHOW_SMOKE_RESULT_SCHEMA = False
SHOW_SMOKE_RAW_RESULTS = False
SMOKE_DISPLAY_MAX_CHARS = 12_000
RUN_EXTRACTION_SMOKE = False

missing_classifications = [
    path for path in CLASSIFICATION_FILES if not path.is_file()
]
if missing_classifications:
    raise FileNotFoundError(
        'Upload the classified inputs or change CLASSIFICATION_ROOT: '
        + ', '.join(map(str, missing_classifications))
    )

for classification_path in CLASSIFICATION_FILES:
    classification_table = pq.read_table(classification_path)
    part_column = (
        'section'
        if 'section' in classification_table.column_names
        else 'part'
    )
    sections = sorted(set(classification_table[part_column].to_pylist()))
    print(
        classification_path.parent.name,
        'paragraphs=', classification_table.num_rows,
        'sections=', sections,
    )

## Load the extraction model

Select **Runtime → Change runtime type → GPU**. Optionally expose a Colab secret named `HF_TOKEN`. The resolved model commit is printed so the smoke run is reproducible.

In [ ]:
import torch
from huggingface_hub import login, model_info

from document_split.config import MODEL_ID
from document_split.runtime import load_extraction_model

if not torch.cuda.is_available():
    raise RuntimeError('A CUDA-enabled Colab runtime is required')

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)

MODEL_REVISION = model_info(MODEL_ID, token=HF_TOKEN).sha
print('Model:', MODEL_ID)
print('Revision:', MODEL_REVISION)
model_pipe, tokenizer = load_extraction_model(
    ACTIVE_V2_EXTRACTION_SETTINGS,
    MODEL_REVISION,
    HF_TOKEN,
)
print('Smoke max new tokens:', SMOKE_MAX_NEW_TOKENS)
print('Model temperature:', MODEL_TEMPERATURE)
print('Sampling enabled:', MODEL_TEMPERATURE > 0)

## Preview routing, batches, and token counts

This prepares the exact map-stage production messages without inference. Each section uses one map call per batch followed by one reduce call over its validated observations and authoritative source paragraphs.

In [ ]:
import pandas as pd
from document_split.processing import count_message_tokens

smoke_preview_rows = []
planned_reduce_keys = set()
for classification_path in CLASSIFICATION_FILES:
    contexts = build_sample_processing_contexts(
        document_id=classification_path.parent.name,
        justice_kind=2,
        parts_parquet_bytes=classification_path.read_bytes(),
        tokenizer=tokenizer,
        chain_settings=SMOKE_CHAIN_SETTINGS,
        part_prompts=DEFAULT_V2_PART_PROCESSING_PROMPTS,
    )
    for processor_name, batches in contexts.items():
        if batches:
            planned_reduce_keys.add((
                classification_path.parent.name,
                processor_name,
            ))
        for batch_number, context in enumerate(batches, start=1):
            smoke_preview_rows.append({
                'document_id': classification_path.parent.name,
                'processor': processor_name,
                'stage': 'map',
                'batch': batch_number,
                'target_paragraphs': len(context.target_paragraph_ids),
                'input_tokens': count_message_tokens(
                    tokenizer, context.messages
                ),
            })

smoke_preview_df = pd.DataFrame(smoke_preview_rows)
display(smoke_preview_df)
PLANNED_MAP_CALLS = len(smoke_preview_df)
PLANNED_REDUCE_CALLS = len(planned_reduce_keys)
PLANNED_MODEL_CALLS = PLANNED_MAP_CALLS + PLANNED_REDUCE_CALLS
print('Planned map calls:', PLANNED_MAP_CALLS)
print('Planned reduce calls:', PLANNED_REDUCE_CALLS)
print('Planned total model calls:', PLANNED_MODEL_CALLS)

## Run isolated inference with live progress

After inspecting the preview, set `RUN_EXTRACTION_SMOKE = True`. The wrapper labels every call as map or reduce and prints a heartbeat once per minute. Optional flags display the exact source chunk, the stage-specific JSON contract, and the raw model output.

In [ ]:
if not RUN_EXTRACTION_SMOKE:
    raise RuntimeError(
        'Review the preview, then set RUN_EXTRACTION_SMOKE = True'
    )

import re
import threading
import time
from document_split.processing import extract_generated_text

class ProgressModelPipe:
    def __init__(self, wrapped, total_calls):
        self.wrapped = wrapped
        self.total_calls = total_calls
        self.started_calls = 0

    def __call__(self, **kwargs):
        self.started_calls += 1
        call_number = self.started_calls
        messages = kwargs.get('text', [])
        user_text = messages[1]['content'][0]['text'] if len(messages) > 1 else ''
        processor = next(
            (
                name
                for name in ('introductory_part', 'reasoning_part', 'operative_part')
                if f'"{name}"' in user_text
            ),
            'unknown_handler',
        )
        target_match = re.search(r'TARGET PARAGRAPH IDS:\n(\[[^\n]+\])', user_text)
        target_ids = target_match.group(1) if target_match else 'unknown'
        stage_match = re.search(r'PROCESSING STAGE:\n([A-Z]+)', user_text)
        stage = stage_match.group(1).lower() if stage_match else 'unknown'
        def message_section(start_marker, end_marker=None):
            if start_marker not in user_text:
                return ''
            section = user_text.split(start_marker, 1)[1]
            if end_marker and end_marker in section:
                section = section.split(end_marker, 1)[0]
            return section.strip()

        if SHOW_SMOKE_CHUNKS:
            chunk_marker = (
                'TARGET PARAGRAPHS:\n'
                if stage == 'map'
                else 'AUTHORITATIVE SOURCE PARAGRAPHS:\n'
            )
            target_chunk = message_section(
                chunk_marker, '\n\nUPSTREAM PARAGRAPH PARTS'
            )
            print(
                f'--- {processor} {stage} source chunk ---\n'
                f'{target_chunk[:SMOKE_DISPLAY_MAX_CHARS]}',
                flush=True,
            )
        if SHOW_SMOKE_RESULT_SCHEMA:
            result_schema = message_section('JSON CONTRACT:\n')
            print(
                f'--- {processor} {stage} JSON result schema ---\n'
                f'{result_schema[:SMOKE_DISPLAY_MAX_CHARS]}',
                flush=True,
            )

        started = time.monotonic()
        print(
            f'[{call_number}/{self.total_calls}] Starting {processor} '
            f'{stage}; '
            f'targets={target_ids}',
            flush=True,
        )
        stop_heartbeat = threading.Event()

        def heartbeat():
            while not stop_heartbeat.wait(60):
                elapsed = time.monotonic() - started
                print(
                    f'[{call_number}/{self.total_calls}] {processor} '
                    f'{stage} still '
                    f'running after {elapsed / 60:.1f} min',
                    flush=True,
                )

        heartbeat_thread = threading.Thread(target=heartbeat, daemon=True)
        heartbeat_thread.start()
        try:
            result = self.wrapped(**kwargs)
            if SHOW_SMOKE_RAW_RESULTS:
                raw_result = extract_generated_text(result)
                print(
                    f'--- {processor} raw model result ---\n'
                    f'{raw_result[:SMOKE_DISPLAY_MAX_CHARS]}',
                    flush=True,
                )
        except Exception:
            elapsed = time.monotonic() - started
            print(
                f'[{call_number}/{self.total_calls}] {processor} {stage} '
                f'failed after '
                f'{elapsed / 60:.1f} min',
                flush=True,
            )
            raise
        finally:
            stop_heartbeat.set()
            heartbeat_thread.join(timeout=1)
        elapsed = time.monotonic() - started
        print(
            f'[{call_number}/{self.total_calls}] Finished {processor} '
            f'{stage} in '
            f'{elapsed / 60:.1f} min',
            flush=True,
        )
        return result

progress_model_pipe = ProgressModelPipe(
    model_pipe,
    total_calls=PLANNED_MODEL_CALLS,
)
print('Artifacts will be written under:', SMOKE_OUTPUT_ROOT, flush=True)
smoke_results = run_real_data_smoke(
    classification_files=CLASSIFICATION_FILES,
    output_root=SMOKE_OUTPUT_ROOT,
    model_pipe=progress_model_pipe,
    tokenizer=tokenizer,
    extraction_settings=ACTIVE_V2_EXTRACTION_SETTINGS,
    chain_settings=SMOKE_CHAIN_SETTINGS,
)
print('Smoke complete. Artifacts:', SMOKE_OUTPUT_ROOT)

## Review population and expected anchors

Population indicates which schema paths received non-empty values. The source-specific checks catch obvious omissions, but passing them does not replace manual comparison with the decision text.

In [ ]:
import json

population_rows = []
smoke_payloads = {}
for smoke_result in smoke_results:
    for schema_path, statistics in smoke_result.population.items():
        population_rows.append({
            'document_id': smoke_result.document_id,
            'path': schema_path,
            **statistics,
        })
    smoke_payloads[smoke_result.document_id] = json.loads(
        (smoke_result.artifact_dir / 'result.json').read_text(
            encoding='utf-8'
        )
    )

population_df = pd.DataFrame(population_rows)
display(
    population_df[population_df['populated']].sort_values(
        ['document_id', 'path']
    )
)

def nested_value(value, dotted_path):
    for key in dotted_path.split('.'):
        if not isinstance(value, dict):
            return None
        value = value.get(key)
    return value

def contains_article(value, article):
    if isinstance(value, dict):
        if str(value.get('article')) == article:
            return True
        return any(
            contains_article(child, article) for child in value.values()
        )
    if isinstance(value, list):
        return any(contains_article(child, article) for child in value)
    return False

EXPECTED_ANCHORS = {
    '118355359': {
        'article': '286',
        'paths': [
            'operative_part.final_sentence',
            'operative_part.probation',
            'operative_part.costs_reimbursement_decision',
            'operative_part.physical_evidence_decision',
        ],
    },
    '118584307': {
        'article': '185',
        'paths': [
            'operative_part.final_sentence',
            'operative_part.sentence_start',
            'operative_part.civil_claim_decision',
        ],
    },
    '116798672': {
        'article': '369-2',
        'paths': [
            'operative_part.final_sentence',
            'operative_part.security_measures_decision',
            'operative_part.physical_evidence_decision',
        ],
    },
}

anchor_rows = []
for smoke_document_id, expected in EXPECTED_ANCHORS.items():
    if smoke_document_id not in smoke_payloads:
        continue
    payload = smoke_payloads[smoke_document_id]
    anchor_rows.append({
        'document_id': smoke_document_id,
        'expectation': f"article {expected['article']}",
        'passed': contains_article(payload, expected['article']),
    })
    for expected_path in expected['paths']:
        extracted_value = nested_value(payload, expected_path)
        anchor_rows.append({
            'document_id': smoke_document_id,
            'expectation': expected_path,
            'passed': extracted_value not in (None, '', []),
        })

anchor_df = pd.DataFrame(anchor_rows)
display(anchor_df)
print('Anchor checks passed:', bool(anchor_df['passed'].all()))

for smoke_document_id, payload in smoke_payloads.items():
    print('\n===', smoke_document_id, '===')
    print(json.dumps(payload, ensure_ascii=False, indent=2)[:12000])

## Download smoke artifacts

Download the ZIP before the Colab runtime is recycled.

In [ ]:
import shutil

smoke_archive = shutil.make_archive(
    '/content/v2-real-data-smoke',
    'zip',
    SMOKE_OUTPUT_ROOT,
)
try:
    from google.colab import files
    files.download(smoke_archive)
except ImportError:
    print('Archive:', smoke_archive)

# Promote validated prompts to the full dataset

Run this section only after the smoke outputs and anchor checks are satisfactory. It uses the existing production V2 pipeline, reading classified paragraph Parquet files from the configured parts bucket and prefix. Results are written under an immutable `info_version_*` destination prefix.

The production runner records a manifest containing prompt, schema, model-revision, and batching hashes. It resumes by skipping completed `result.parquet` objects. If an existing manifest differs, the run stops and requires a new info version instead of mixing incompatible results.

## Configure the production run

Use a new `FULL_INFO_VERSION` whenever prompts, schema, model, or batching settings change. `FULL_LIMIT = None` processes the complete eligible dataset; use a small integer for a cloud staging run.

In [ ]:
from dataclasses import replace

from document_split.v2 import (
    DEFAULT_V2_CHAIN_SETTINGS,
    DEFAULT_V2_STORAGE_SETTINGS,
    run_v2_pipeline,
)

# Choose a new immutable output namespace for this prompt/schema version.
FULL_INFO_VERSION = 'info_version_13'
FULL_JUSTICE_KINDS = (2,)
FULL_LIMIT = None  # None means the complete eligible dataset.
FULL_SKIP_EXISTING = True

FULL_STORAGE_SETTINGS = replace(
    DEFAULT_V2_STORAGE_SETTINGS,
    info_version=FULL_INFO_VERSION,
    justice_kinds=FULL_JUSTICE_KINDS,
    limit=FULL_LIMIT,
    skip_existing=FULL_SKIP_EXISTING,
)
# Promote the batching values validated by the smoke run.
FULL_CHAIN_SETTINGS = replace(
    DEFAULT_V2_CHAIN_SETTINGS,
    target_batch_tokens=SMOKE_TARGET_BATCH_TOKENS,
    overlap_tokens=SMOKE_OVERLAP_TOKENS,
)

# Both safeguards must be changed deliberately.
RUN_FULL_DATASET = False
FULL_RUN_CONFIRMATION = ''  # Set exactly to: RUN FULL DATASET

print('Input bucket:', FULL_STORAGE_SETTINGS.parts_bucket)
print('Input prefix:', FULL_STORAGE_SETTINGS.parts_prefix)
print('Output bucket:', FULL_STORAGE_SETTINGS.destination_bucket)
print('Output version:', FULL_STORAGE_SETTINGS.version_prefix)
print('Justice kinds:', FULL_STORAGE_SETTINGS.justice_kinds)
print('Limit:', FULL_STORAGE_SETTINGS.limit)
print('Skip existing:', FULL_STORAGE_SETTINGS.skip_existing)

## Start or resume full processing

The pipeline reads the Colab secrets `cloud_access` and, when needed, `HF_TOKEN`. The smoke model is released first so the production runner can load the manifest-pinned model without duplicating GPU memory.

In [ ]:
if not RUN_FULL_DATASET:
    raise RuntimeError('Set RUN_FULL_DATASET = True after validation')
if FULL_RUN_CONFIRMATION != 'RUN FULL DATASET':
    raise RuntimeError(
        "Set FULL_RUN_CONFIRMATION exactly to 'RUN FULL DATASET'"
    )
if FULL_STORAGE_SETTINGS.limit is not None:
    print(
        'Staging mode: at most',
        FULL_STORAGE_SETTINGS.limit,
        'eligible documents will be considered.',
    )
else:
    print('Full mode: processing every eligible unfinished document.')

# Avoid holding two copies of the model on the GPU.
if 'model_pipe' in globals():
    del model_pipe
if 'tokenizer' in globals():
    del tokenizer
import gc
gc.collect()
torch.cuda.empty_cache()

full_run_result = run_v2_pipeline(
    extraction_settings=ACTIVE_V2_EXTRACTION_SETTINGS,
    storage_settings=FULL_STORAGE_SETTINGS,
    chain_settings=FULL_CHAIN_SETTINGS,
    part_prompts=DEFAULT_V2_PART_PROCESSING_PROMPTS,
)
print('Full-run counters:', full_run_result)